# POC — Short-horizon tree models with fixed ATR-unit handling

## Experiment recap

This notebook evaluates whether the current point-in-time feature set can identify a small,
repeatable short-horizon edge over **1–5 observed trading sessions**.

The experiment compares:

- Ridge regression as a linear baseline;
- `XGBRegressor` for nonlinear return prediction;
- `XGBRanker` for within-date candidate ordering;
- raw adjusted-close forward returns; and
- forward price changes normalized by signal-date ATR.

The important target correction is that all prices used by the ATR-unit target live in the same
split-adjusted price space. Historical rows with zero volume that exactly duplicate the previous
OHLC bar are provider placeholders rather than observed trading sessions. They are removed before
ATR and forward-session shifts, so they cannot artificially decay ATR or count toward the horizon.

The canonical temporal dataset remains authoritative for features, sample alignment, and fixed
train/validation/test boundaries. This notebook generates replacement short-horizon targets and
aligns them back to that canonical sample index. The locked test split is not evaluated.

## 1. Imports and configuration

In [ ]:
from datetime import date
import gc
import importlib.metadata
from itertools import product
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, pearsonr, spearmanr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import text
from tqdm.auto import tqdm
from xgboost import XGBRanker, XGBRegressor

from swingtrader.data.db import resolve_database_engine
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import (
    UniverseSpec,
    FORWARD_RETURN_PRIMARY_TASK,
    FORWARD_RETURN_TARGET_SET,
    build_temporal_dataset,
)
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    FixedTemporalSplitter,
    ModelSpec,
    TemporalSplitSpec,
)
from swingtrader.modeling.training import LOGISTIC_REGRESSION_MODEL_TYPE


PROVIDER = "yfinance"

# Example smoke test: ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
TICKERS: tuple[str, ...] | None = None

DATA_CUTOFF = date(2026, 7, 24)

TRAIN_START = date(2000, 1, 1)
TRAIN_END = date(2022, 12, 31)
VALIDATION_START = date(2023, 1, 1)
VALIDATION_END = date(2024, 12, 31)
TEST_START = date(2025, 1, 1)
TEST_END = DATA_CUTOFF

HORIZONS = (1, 2, 3, 4, 5)
TARGET_FAMILIES = ("raw_return", "atr_units")
MODEL_NAMES = ("ridge", "xgboost_regressor", "xgboost_ranker")
TOP_K_VALUES = (1, 3, 5)

ATR_LENGTH = 14
MIN_ATR_FRACTION_OF_PRICE = 1e-4
MAX_ABS_ATR_UNITS = 50.0
MAX_ABS_RAW_RETURN = 3.0
FLOAT_ATOL = 1e-10

# Relevance grades 0–4. The highest grade represents approximately the top 3% of each date.
RANK_RELEVANCE_QUANTILES = (0.50, 0.75, 0.90, 0.97)

RANDOM_SEED = 42
XGB_N_JOBS = -1
STORE_FITTED_MODELS = False
MAX_SHORTLIST_CONFIGURATIONS = 5

# Resampling is intentionally opt-in. Use small counts while iterating, then increase finalists.
RUN_RESAMPLING = False
BOOTSTRAP_ITERATIONS = 500
PERMUTATION_ITERATIONS = 250
RESAMPLING_CONFIGURATION_LIMIT = 3

XGB_REGRESSOR_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 3,
    "gamma": 0.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "n_jobs": XGB_N_JOBS,
    "random_state": RANDOM_SEED,
}

XGB_RANKER_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 3,
    "gamma": 0.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "objective": "rank:ndcg",
    "eval_metric": "ndcg@5",
    "tree_method": "hist",
    "n_jobs": XGB_N_JOBS,
    "random_state": RANDOM_SEED,
}

### Runtime policy

The 30 model runs are executed sequentially with a progress bar. Both XGBoost estimators use
multiple CPU threads internally, so outer multiprocessing would normally oversubscribe the
machine rather than accelerate this matrix. Resampling is consolidated across top-k values and
is disabled until the validation shortlist exists.

## 2. Repository, database, and canonical baseline bundle

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory.")


repo_root = find_repo_root(Path.cwd())
database_path = repo_root / "data" / "swingtrader.sqlite"
database_url = f"sqlite+pysqlite:///{database_path.as_posix()}"
engine = resolve_database_engine(database_url=database_url)

if TICKERS is None:
    with engine.connect() as connection:
        ticker_rows = connection.execute(
            text(
                '''
                SELECT DISTINCT ticker
                FROM bronze_market_daily_prices
                WHERE provider = :provider
                ORDER BY ticker
                '''
            ),
            {"provider": PROVIDER},
        ).fetchall()
    resolved_tickers = tuple(row[0] for row in ticker_rows)
else:
    resolved_tickers = tuple(TICKERS)

if not resolved_tickers:
    raise RuntimeError(
        f"No bronze tickers were found for provider {PROVIDER!r}. "
        "Populate the local database or set TICKERS explicitly."
    )

package_versions = {}
for package in ("numpy", "pandas", "scikit-learn", "xgboost"):
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = "not installed"

{
    "database_path": database_path,
    "ticker_count": len(resolved_tickers),
    "first_tickers": resolved_tickers[:10],
    "package_versions": package_versions,
}

In [ ]:
feature_set = DEFAULT_FEATURE_SET

placeholder_model = ModelSpec(
    name="short_horizon_tree_model_poc",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters={},
    feature_columns=None,
)

universe = UniverseSpec(
    name="local_bronze_short_horizon_universe",
    version="1",
    provider=PROVIDER,
    tickers=resolved_tickers,
)

split_spec = TemporalSplitSpec(
    name="short_horizon_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

experiment_spec = ExperimentSpec(
    name="short_horizon_tree_models",
    version="1",
    feature_set=feature_set,
    target_set=FORWARD_RETURN_TARGET_SET,
    task=FORWARD_RETURN_PRIMARY_TASK,
    universe=universe,
    data_start=TRAIN_START,
    data_end=TEST_END,
    split=split_spec,
    model=placeholder_model,
    random_seeds={"model": RANDOM_SEED, "evaluation": RANDOM_SEED + 1},
)

bundle = build_temporal_dataset(engine=engine, spec=experiment_spec.dataset_spec)
split_result = FixedTemporalSplitter(experiment_spec.split).assign(bundle)

if not bundle.features.index.equals(bundle.samples.index):
    raise ValueError("Canonical feature and sample indexes are not aligned.")
if not bundle.features.index.is_unique:
    raise ValueError("Canonical bundle index must be unique.")

{
    "experiment_digest": experiment_spec.digest,
    "ticker_count": len(resolved_tickers),
    "generated_feature_count": len(bundle.manifest.feature_columns),
    "outer_train": split_result.summary("train").to_manifest(),
    "outer_validation": split_result.summary("validation").to_manifest(),
}

The canonical bundle supplies features, sample metadata, and the fixed outer split. The notebook also records the actual end date of every cleaned-session target and reapplies
the train and validation boundaries explicitly. No test
positions are requested anywhere in this notebook.

## 3. Load and clean target-source OHLCV

In [ ]:
INDEX_LEVELS = ["provider", "ticker", "trading_date"]
GROUP_LEVELS = ["provider", "ticker"]
OHLC_COLUMNS = ["open", "high", "low", "close"]

ticker_placeholders = ", ".join(f":ticker_{index}" for index in range(len(resolved_tickers)))
query_parameters = {
    "provider": PROVIDER,
    "data_cutoff": DATA_CUTOFF.isoformat(),
    **{
        f"ticker_{index}": ticker
        for index, ticker in enumerate(resolved_tickers)
    },
}

price_query = text(
    f'''
    SELECT
        provider,
        ticker,
        trading_date,
        open,
        high,
        low,
        close,
        adjusted_close,
        volume
    FROM bronze_market_daily_prices
    WHERE provider = :provider
      AND ticker IN ({ticker_placeholders})
      AND trading_date <= :data_cutoff
    ORDER BY provider, ticker, trading_date
    '''
)

with engine.connect() as connection:
    prices = pd.read_sql_query(
        price_query,
        connection,
        params=query_parameters,
        parse_dates=["trading_date"],
    )

if prices.empty:
    raise RuntimeError("No bronze OHLCV rows were loaded for target construction.")

for column in [*OHLC_COLUMNS, "adjusted_close", "volume"]:
    prices[column] = pd.to_numeric(prices[column], errors="coerce")

prices = (
    prices.set_index(INDEX_LEVELS)
    .sort_index()
)

if not prices.index.is_unique:
    duplicate_count = int(prices.index.duplicated(keep=False).sum())
    raise ValueError(f"Bronze OHLCV index contains {duplicate_count:,} duplicate rows.")

previous_ohlc = (
    prices.loc[:, OHLC_COLUMNS]
    .groupby(level=GROUP_LEVELS, sort=False)
    .shift(1)
)

duplicated_previous_ohlc = pd.Series(True, index=prices.index)
for column in OHLC_COLUMNS:
    duplicated_previous_ohlc &= np.isclose(
        prices[column],
        previous_ohlc[column],
        rtol=0.0,
        atol=FLOAT_ATOL,
        equal_nan=False,
    )

zero_volume = prices["volume"].eq(0)
synthetic_no_trade_bar = zero_volume & duplicated_previous_ohlc
zero_volume_price_changed = zero_volume & ~duplicated_previous_ohlc

placeholder_audit = prices.loc[
    synthetic_no_trade_bar | zero_volume_price_changed,
    [*OHLC_COLUMNS, "adjusted_close", "volume"],
].copy()
for column in OHLC_COLUMNS:
    placeholder_audit[f"previous_{column}"] = previous_ohlc.loc[
        placeholder_audit.index,
        column,
    ]
placeholder_audit["reason"] = np.where(
    synthetic_no_trade_bar.loc[placeholder_audit.index],
    "zero_volume_duplicated_previous_ohlc",
    "zero_volume_price_changed",
)

placeholder_summary = pd.DataFrame(
    {
        "row_count": [
            int(synthetic_no_trade_bar.sum()),
            int(zero_volume_price_changed.sum()),
        ],
        "ticker_count": [
            int(
                placeholder_audit.loc[
                    placeholder_audit["reason"].eq(
                        "zero_volume_duplicated_previous_ohlc"
                    )
                ]
                .reset_index()["ticker"]
                .nunique()
            ),
            int(
                placeholder_audit.loc[
                    placeholder_audit["reason"].eq("zero_volume_price_changed")
                ]
                .reset_index()["ticker"]
                .nunique()
            ),
        ],
    },
    index=[
        "removed_zero_volume_duplicated_previous_ohlc",
        "retained_zero_volume_price_changed",
    ],
)

clean_prices = prices.loc[~synthetic_no_trade_bar].copy()

display(placeholder_summary)
if zero_volume_price_changed.any():
    print(
        "Zero-volume rows with changed prices are retained and shown for audit. "
        "They are not automatically assumed to be placeholders."
    )
    display(
        placeholder_audit.loc[
            placeholder_audit["reason"].eq("zero_volume_price_changed")
        ].head(100)
    )

print(
    f"Removed {int(synthetic_no_trade_bar.sum()):,} synthetic no-trade rows; "
    f"{len(clean_prices):,} observed rows remain for ATR and horizon construction."
)

## 4. Construct aligned 1–5 session targets

In [ ]:
def grouped_shift(series: pd.Series, periods: int) -> pd.Series:
    return series.groupby(level=GROUP_LEVELS, sort=False).shift(periods)


raw_close = clean_prices["close"]
adjusted_close = clean_prices["adjusted_close"]

adjustment_factor = adjusted_close.div(raw_close)
valid_adjustment_factor = (
    np.isfinite(adjustment_factor)
    & adjustment_factor.gt(0)
    & raw_close.gt(0)
    & adjusted_close.gt(0)
)

target_source = clean_prices.copy()
target_source["adjustment_factor"] = adjustment_factor.where(valid_adjustment_factor)
target_source["adjusted_open"] = (
    target_source["open"] * target_source["adjustment_factor"]
)
target_source["adjusted_high"] = (
    target_source["high"] * target_source["adjustment_factor"]
)
target_source["adjusted_low"] = (
    target_source["low"] * target_source["adjustment_factor"]
)
target_source["adjusted_close_for_targets"] = adjusted_close.where(
    valid_adjustment_factor
)

adjusted_price_columns = [
    "adjusted_open",
    "adjusted_high",
    "adjusted_low",
    "adjusted_close_for_targets",
]
finite_adjusted_prices = np.isfinite(target_source[adjusted_price_columns]).all(axis=1)
positive_adjusted_prices = target_source[adjusted_price_columns].gt(0).all(axis=1)
consistent_adjusted_range = (
    target_source["adjusted_high"].ge(target_source["adjusted_low"])
    & target_source["adjusted_high"].ge(target_source["adjusted_open"])
    & target_source["adjusted_high"].ge(target_source["adjusted_close_for_targets"])
    & target_source["adjusted_low"].le(target_source["adjusted_open"])
    & target_source["adjusted_low"].le(target_source["adjusted_close_for_targets"])
)
target_source["valid_adjusted_bar"] = (
    valid_adjustment_factor
    & finite_adjusted_prices
    & positive_adjusted_prices
    & consistent_adjusted_range
)

previous_adjusted_close = grouped_shift(
    target_source["adjusted_close_for_targets"],
    1,
)
true_range = pd.concat(
    [
        target_source["adjusted_high"] - target_source["adjusted_low"],
        (target_source["adjusted_high"] - previous_adjusted_close).abs(),
        (target_source["adjusted_low"] - previous_adjusted_close).abs(),
    ],
    axis=1,
).max(axis=1, skipna=True)
true_range = true_range.where(target_source["valid_adjusted_bar"])

atr_column = f"atr_{ATR_LENGTH}"
atr_fraction_column = f"atr_fraction_{ATR_LENGTH}"

target_source[atr_column] = true_range.groupby(
    level=GROUP_LEVELS,
    sort=False,
).transform(
    lambda values: values.ewm(
        alpha=1.0 / ATR_LENGTH,
        adjust=False,
        min_periods=ATR_LENGTH,
    ).mean()
)
target_source[atr_fraction_column] = (
    target_source[atr_column]
    / target_source["adjusted_close_for_targets"]
)
target_source["valid_atr"] = (
    target_source["valid_adjusted_bar"]
    & np.isfinite(target_source[atr_column])
    & target_source[atr_column].gt(0)
    & np.isfinite(target_source[atr_fraction_column])
    & target_source[atr_fraction_column].ge(MIN_ATR_FRACTION_OF_PRICE)
)

zero_adjusted_range = (
    target_source["adjusted_high"]
    .sub(target_source["adjusted_low"])
    .abs()
    .le(FLOAT_ATOL)
)
unchanged_adjusted_close = (
    target_source["adjusted_close_for_targets"]
    .sub(previous_adjusted_close)
    .abs()
    .le(FLOAT_ATOL)
)
target_source[f"zero_range_fraction_{ATR_LENGTH}"] = (
    zero_adjusted_range.astype("float64")
    .groupby(level=GROUP_LEVELS, sort=False)
    .transform(
        lambda values: values.rolling(
            ATR_LENGTH,
            min_periods=ATR_LENGTH,
        ).mean()
    )
)
target_source[f"unchanged_close_fraction_{ATR_LENGTH}"] = (
    unchanged_adjusted_close.astype("float64")
    .groupby(level=GROUP_LEVELS, sort=False)
    .transform(
        lambda values: values.rolling(
            ATR_LENGTH,
            min_periods=ATR_LENGTH,
        ).mean()
    )
)

trading_date_series = pd.Series(
    pd.DatetimeIndex(target_source.index.get_level_values("trading_date")),
    index=target_source.index,
    name="trading_date",
)

raw_target_columns = []
atr_target_columns = []
target_end_columns = []
extreme_target_frames = []

for horizon in HORIZONS:
    future_adjusted_close = grouped_shift(
        target_source["adjusted_close_for_targets"],
        -horizon,
    )
    valid_future_price = (
        np.isfinite(future_adjusted_close)
        & future_adjusted_close.gt(0)
    )

    raw_column = f"raw_return_{horizon}d"
    atr_column_name = f"atr_units_{horizon}d"
    target_end_column = f"target_end_{horizon}d"
    raw_target_columns.append(raw_column)
    atr_target_columns.append(atr_column_name)
    target_end_columns.append(target_end_column)
    target_source[target_end_column] = grouped_shift(
        trading_date_series,
        -horizon,
    )

    raw_values = (
        future_adjusted_close
        / target_source["adjusted_close_for_targets"]
        - 1.0
    )
    raw_values = raw_values.where(
        target_source["valid_adjusted_bar"] & valid_future_price
    )

    atr_values = (
        future_adjusted_close
        - target_source["adjusted_close_for_targets"]
    ) / target_source[atr_column]
    atr_values = atr_values.where(
        target_source["valid_atr"] & valid_future_price
    )

    extreme_raw = raw_values.abs().gt(MAX_ABS_RAW_RETURN)
    extreme_atr = atr_values.abs().gt(MAX_ABS_ATR_UNITS)

    if extreme_raw.any():
        rows = target_source.loc[
            extreme_raw,
            [
                "adjustment_factor",
                "adjusted_close_for_targets",
                atr_column,
                atr_fraction_column,
                f"zero_range_fraction_{ATR_LENGTH}",
                f"unchanged_close_fraction_{ATR_LENGTH}",
                "volume",
            ],
        ].copy()
        rows["target_column"] = raw_column
        rows["target_value"] = raw_values.loc[extreme_raw]
        rows["exclusion_reason"] = "absolute_raw_return_above_guard"
        extreme_target_frames.append(rows)

    if extreme_atr.any():
        rows = target_source.loc[
            extreme_atr,
            [
                "adjustment_factor",
                "adjusted_close_for_targets",
                atr_column,
                atr_fraction_column,
                f"zero_range_fraction_{ATR_LENGTH}",
                f"unchanged_close_fraction_{ATR_LENGTH}",
                "volume",
            ],
        ].copy()
        rows["target_column"] = atr_column_name
        rows["target_value"] = atr_values.loc[extreme_atr]
        rows["exclusion_reason"] = "absolute_atr_units_above_guard"
        extreme_target_frames.append(rows)

    target_source[raw_column] = raw_values.mask(extreme_raw)
    target_source[atr_column_name] = atr_values.mask(extreme_atr)

extreme_target_audit = (
    pd.concat(extreme_target_frames).sort_values(
        "target_value",
        key=lambda values: values.abs(),
        ascending=False,
    )
    if extreme_target_frames
    else pd.DataFrame()
)

target_columns = raw_target_columns + atr_target_columns
short_targets = target_source.reindex(bundle.features.index)

if not short_targets.index.equals(bundle.features.index):
    raise ValueError("Short-horizon targets do not align with the canonical bundle index.")

remaining_extreme_raw = short_targets[raw_target_columns].abs().gt(
    MAX_ABS_RAW_RETURN
).to_numpy().any()
remaining_extreme_atr = short_targets[atr_target_columns].abs().gt(
    MAX_ABS_ATR_UNITS
).to_numpy().any()
if remaining_extreme_raw or remaining_extreme_atr:
    raise ValueError("Target sanitation failed to remove configured extreme values.")

In [ ]:
target_quality_summary = {
    "bronze_source_rows": len(prices),
    "removed_placeholder_rows": int(synthetic_no_trade_bar.sum()),
    "clean_observed_rows": len(clean_prices),
    "canonical_bundle_rows": len(bundle.features),
    "aligned_rows_with_any_target": int(
        short_targets[target_columns].notna().any(axis=1).sum()
    ),
    "invalid_adjusted_bar_fraction": float(
        (~short_targets["valid_adjusted_bar"].fillna(False)).mean()
    ),
    "invalid_atr_fraction": float(
        (~short_targets["valid_atr"].fillna(False)).mean()
    ),
    "atr_missing_fraction": float(short_targets[atr_column].isna().mean()),
    "atr_fraction_distribution": (
        short_targets[atr_fraction_column]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .describe(percentiles=[0.001, 0.01, 0.50, 0.99, 0.999])
        .to_dict()
    ),
    "adjustment_factor_distribution": (
        short_targets["adjustment_factor"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .describe(percentiles=[0.001, 0.01, 0.50, 0.99, 0.999])
        .to_dict()
    ),
    "target_missing_fractions": (
        short_targets[target_columns]
        .isna()
        .mean()
        .to_dict()
    ),
    "excluded_extreme_target_rows": len(extreme_target_audit),
}
target_quality_summary

In [ ]:
target_distribution = short_targets[target_columns].describe(
    percentiles=[0.001, 0.01, 0.50, 0.99, 0.999]
).T
display(target_distribution)

if extreme_target_audit.empty:
    print("No targets exceeded the broad raw-return or ATR-unit sanity guards.")
else:
    print(
        f"Excluded {len(extreme_target_audit):,} extreme target observations. "
        "These rows remain available below for manual verification."
    )
    display(extreme_target_audit.head(100))

### Target semantics

- `raw_return_hd` is the adjusted-close forward return over `h` observed sessions.
- `atr_units_hd` is the adjusted-close change divided by split-adjusted ATR at the signal date.
- Open, high, low, and close are transformed into the same adjusted price space before true range
  and ATR are calculated.
- Confirmed zero-volume duplicated-OHLC placeholders are removed before ATR and session shifts.
- Zero-volume rows whose prices changed are retained and surfaced for audit.
- Zero, non-finite, or implausibly tiny ATR denominators are rejected rather than replaced by a
  microscopic floor.
- Very broad raw-return and ATR-unit guards convert residual extreme targets to missing values and
  preserve the excluded rows in `extreme_target_audit`.
- Each target records its actual cleaned-session end date; train and validation rows are filtered so the target cannot cross the split boundary.

## 5. Extract train and validation frames

In [ ]:
DATE_LEVEL = "trading_date"
TICKER_LEVEL = "ticker"

feature_columns = tuple(bundle.manifest.feature_columns)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

X_all = bundle.features.loc[:, feature_columns]
metadata_all = bundle.samples


def target_column(target_family: str, horizon: int) -> str:
    if target_family not in TARGET_FAMILIES:
        raise KeyError(f"Unknown target family: {target_family!r}")
    return f"{target_family}_{horizon}d"


def make_experiment_frames(target_family: str, horizon: int):
    selected_target = target_column(target_family, horizon)
    raw_return_column = f"raw_return_{horizon}d"

    target_end_column = f"target_end_{horizon}d"

    def build(positions, *, split_end: date):
        X = X_all.iloc[positions].copy()
        metadata = metadata_all.iloc[positions].copy()
        targets = short_targets.iloc[positions]

        y = pd.to_numeric(targets[selected_target], errors="coerce")
        actual_return = pd.to_numeric(targets[raw_return_column], errors="coerce")
        target_end = pd.to_datetime(targets[target_end_column], errors="coerce")
        valid = (
            y.notna()
            & actual_return.notna()
            & target_end.notna()
            & np.isfinite(y)
            & np.isfinite(actual_return)
            & target_end.le(pd.Timestamp(split_end))
        )

        X = X.loc[valid]
        y = y.loc[valid].astype("float64")
        frame = metadata.loc[valid].copy()
        frame["actual_target"] = y
        frame["actual_return"] = actual_return.loc[valid].astype("float64")
        return X, y, frame

    return (
        *build(train_positions, split_end=TRAIN_END),
        *build(validation_positions, split_end=VALIDATION_END),
    )


shape_rows = []
for family in TARGET_FAMILIES:
    for horizon in HORIZONS:
        X_train, y_train, _, X_validation, y_validation, _ = make_experiment_frames(
            family,
            horizon,
        )
        shape_rows.append(
            {
                "target_family": family,
                "horizon": horizon,
                "train_rows": len(y_train),
                "validation_rows": len(y_validation),
                "feature_count": X_train.shape[1],
                "train_target_mean": y_train.mean(),
                "validation_target_mean": y_validation.mean(),
            }
        )

pd.DataFrame(shape_rows).set_index(["target_family", "horizon"])

## 6. Models and learning-to-rank labels

In [ ]:
def make_model(model_name: str):
    if model_name == "ridge":
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("regressor", Ridge(alpha=10.0)),
            ]
        )
    if model_name == "xgboost_regressor":
        return XGBRegressor(**XGB_REGRESSOR_PARAMS)
    if model_name == "xgboost_ranker":
        return XGBRanker(**XGB_RANKER_PARAMS)
    raise KeyError(f"Unknown model: {model_name!r}")


def date_values(frame: pd.DataFrame) -> pd.DatetimeIndex:
    if DATE_LEVEL in frame.index.names:
        return pd.DatetimeIndex(frame.index.get_level_values(DATE_LEVEL))
    if DATE_LEVEL in frame.columns:
        return pd.DatetimeIndex(frame[DATE_LEVEL])
    raise KeyError(f"{DATE_LEVEL!r} is unavailable in the frame.")


def ticker_values(frame: pd.DataFrame) -> np.ndarray:
    if TICKER_LEVEL in frame.index.names:
        return frame.index.get_level_values(TICKER_LEVEL).astype(str).to_numpy()
    if TICKER_LEVEL in frame.columns:
        return frame[TICKER_LEVEL].astype(str).to_numpy()
    return np.arange(len(frame)).astype(str)


def relevance_labels(target: pd.Series, metadata: pd.DataFrame) -> pd.Series:
    date_groups = pd.Series(
        date_values(metadata).to_numpy(),
        index=target.index,
        name="_date_group",
    )
    percentiles = target.groupby(date_groups, sort=False).rank(
        method="average",
        pct=True,
    )
    labels = np.searchsorted(
        np.asarray(RANK_RELEVANCE_QUANTILES, dtype="float64"),
        percentiles.to_numpy(dtype="float64"),
        side="right",
    )
    return pd.Series(labels.astype("int32"), index=target.index, name="relevance")


def ranking_training_inputs(
    X: pd.DataFrame,
    target: pd.Series,
    metadata: pd.DataFrame,
):
    dates = date_values(metadata)
    tickers = ticker_values(metadata)
    order = np.lexsort((tickers, dates.asi8))

    X_sorted = X.iloc[order]
    labels_sorted = relevance_labels(target, metadata).iloc[order]
    sorted_dates = dates[order]
    qid = pd.factorize(sorted_dates, sort=False)[0].astype("int32")

    if np.any(np.diff(qid) < 0):
        raise ValueError("Ranking query identifiers must be sorted in non-decreasing order.")
    return X_sorted, labels_sorted, qid


def fit_and_predict(
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    train_metadata: pd.DataFrame,
    X_validation: pd.DataFrame,
):
    model = make_model(model_name)
    if model_name == "xgboost_ranker":
        X_rank, labels, qid = ranking_training_inputs(X_train, y_train, train_metadata)
        model.fit(X_rank, labels, qid=qid)
    else:
        model.fit(X_train, y_train)

    predictions = np.asarray(model.predict(X_validation), dtype="float64").reshape(-1)
    if len(predictions) != len(X_validation):
        raise ValueError("Model prediction count does not match validation rows.")
    return model, predictions

In [ ]:
# Inspect the graded relevance distribution before fitting any ranker.
_, example_target, example_metadata, _, _, _ = make_experiment_frames("raw_return", 5)
example_relevance = relevance_labels(example_target, example_metadata)
example_relevance.value_counts(normalize=True).sort_index().rename("fraction")

XGBRanker treats every signal date as one query group. Its labels are graded relevance levels,
not the continuous return itself. This lets the ranker optimize within-date ordering while all
models are still evaluated against the same continuous realized target and raw return.

## 7. Evaluation and tie-aware selection helpers

In [ ]:
def group_by_date(frame: pd.DataFrame, *, sort: bool = True):
    if DATE_LEVEL in frame.index.names:
        return frame.groupby(level=DATE_LEVEL, sort=sort)
    if DATE_LEVEL in frame.columns:
        return frame.groupby(DATE_LEVEL, sort=sort)
    raise KeyError(f"{DATE_LEVEL!r} is unavailable in the frame.")


def finite_correlation(function, x, y) -> float:
    x_array = np.asarray(x, dtype="float64")
    y_array = np.asarray(y, dtype="float64")
    mask = np.isfinite(x_array) & np.isfinite(y_array)
    if mask.sum() < 3:
        return np.nan
    if np.ptp(x_array[mask]) == 0 or np.ptp(y_array[mask]) == 0:
        return np.nan
    result = function(x_array[mask], y_array[mask])
    return float(result.statistic if hasattr(result, "statistic") else result[0])


def add_predictions(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    prediction_array = np.asarray(predictions, dtype="float64").reshape(-1)
    if len(prediction_array) != len(frame):
        raise ValueError("Prediction count does not match frame rows.")

    result = frame.copy()
    result["predicted_score"] = prediction_array
    return result


def daily_ranking_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        actual = group["actual_target"]
        predicted = group["predicted_score"]
        percentiles = actual.rank(method="average", pct=True)
        relevance = np.searchsorted(
            np.asarray(RANK_RELEVANCE_QUANTILES, dtype="float64"),
            percentiles.to_numpy(dtype="float64"),
            side="right",
        )
        ndcg = np.nan
        if len(group) >= 2 and np.max(relevance) > 0:
            ndcg = float(
                ndcg_score(
                    relevance.reshape(1, -1),
                    predicted.to_numpy(dtype="float64").reshape(1, -1),
                    k=min(5, len(group)),
                )
            )
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(group),
                "spearman": finite_correlation(spearmanr, predicted, actual),
                "kendall_tau": finite_correlation(kendalltau, predicted, actual),
                "ndcg_at_5": ndcg,
            }
        )
    return pd.DataFrame(rows)


def prediction_cardinality_by_date(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        scores = group["predicted_score"]
        unique_predictions = int(scores.nunique())
        maximum = scores.max()
        top_bucket_size = int(scores.eq(maximum).sum())
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(group),
                "unique_predictions": unique_predictions,
                "top_bucket_size": top_bucket_size,
                "flat_score_date": unique_predictions == 1,
                "prediction_std": float(scores.std(ddof=0)),
            }
        )
    return pd.DataFrame(rows).set_index(DATE_LEVEL)


def aggregate_metrics(frame: pd.DataFrame, *, model_name: str) -> dict[str, float]:
    valid = np.isfinite(frame["actual_target"]) & np.isfinite(frame["predicted_score"])
    evaluated = frame.loc[valid]
    actual = evaluated["actual_target"].to_numpy(dtype="float64")
    predicted = evaluated["predicted_score"].to_numpy(dtype="float64")
    daily = daily_ranking_metrics(evaluated)
    cardinality = prediction_cardinality_by_date(evaluated)

    regression_metrics = {"mae": np.nan, "rmse": np.nan, "r2": np.nan}
    if model_name != "xgboost_ranker":
        regression_metrics = {
            "mae": mean_absolute_error(actual, predicted),
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "r2": r2_score(actual, predicted),
        }

    return {
        "rows": len(evaluated),
        **regression_metrics,
        "pearson": finite_correlation(pearsonr, predicted, actual),
        "pooled_spearman": finite_correlation(spearmanr, predicted, actual),
        "pooled_kendall_tau": finite_correlation(kendalltau, predicted, actual),
        "mean_daily_spearman": daily["spearman"].mean(),
        "mean_daily_kendall_tau": daily["kendall_tau"].mean(),
        "mean_daily_ndcg_at_5": daily["ndcg_at_5"].mean(),
        "prediction_mean": predicted.mean(),
        "prediction_std": predicted.std(),
        "global_unique_predictions": np.unique(predicted).size,
        "median_daily_unique_predictions": cardinality["unique_predictions"].median(),
        "flat_score_date_fraction": cardinality["flat_score_date"].mean(),
        "median_top_bucket_size": cardinality["top_bucket_size"].median(),
    }

In [ ]:
def eligible_daily_group(group: pd.DataFrame) -> pd.DataFrame:
    valid = (
        np.isfinite(group["predicted_score"])
        & np.isfinite(group["actual_target"])
        & np.isfinite(group["actual_return"])
    )
    return group.loc[valid].copy()


def deterministic_order(group: pd.DataFrame) -> pd.DataFrame:
    ordered = group.copy()
    ordered["_ticker_sort"] = ticker_values(ordered)
    return ordered.sort_values(
        ["predicted_score", "_ticker_sort"],
        ascending=[False, True],
        kind="mergesort",
    ).drop(columns="_ticker_sort")


def select_daily_top(
    frame: pd.DataFrame,
    k: int,
    *,
    skip_flat_dates: bool = True,
) -> pd.DataFrame:
    selected = []
    for _, group in group_by_date(frame, sort=True):
        eligible = eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_score"].nunique() == 1:
            continue
        selected.append(deterministic_order(eligible).head(min(k, len(eligible))))

    if not selected:
        return frame.iloc[0:0].copy()
    return pd.concat(selected)


def select_daily_top_score_bucket(
    frame: pd.DataFrame,
    *,
    skip_flat_dates: bool = True,
) -> pd.DataFrame:
    selected = []
    for _, group in group_by_date(frame, sort=True):
        eligible = eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_score"].nunique() == 1:
            continue
        selected.append(eligible.loc[eligible["predicted_score"].eq(
            eligible["predicted_score"].max()
        )])

    if not selected:
        return frame.iloc[0:0].copy()
    return pd.concat(selected)


def selected_rows(frame: pd.DataFrame, selection: str) -> pd.DataFrame:
    if selection == "top_score_bucket":
        return select_daily_top_score_bucket(frame, skip_flat_dates=True)
    if selection.startswith("top_"):
        return select_daily_top(
            frame,
            int(selection.removeprefix("top_")),
            skip_flat_dates=True,
        )
    raise KeyError(f"Unknown selection: {selection!r}")


def daily_selection_outcomes(frame: pd.DataFrame, selection: str) -> pd.DataFrame:
    selected = selected_rows(frame, selection)
    if selected.empty:
        return pd.DataFrame(
            columns=[
                "selected_return",
                "selected_target",
                "selected_count",
                "universe_return",
                "return_spread",
            ]
        )

    selected_daily = group_by_date(selected, sort=True).agg(
        selected_return=("actual_return", "mean"),
        selected_target=("actual_target", "mean"),
        selected_count=("actual_return", "size"),
    )
    universe_daily = group_by_date(frame, sort=True)["actual_return"].mean()
    selected_daily["universe_return"] = universe_daily.reindex(selected_daily.index)
    selected_daily["return_spread"] = (
        selected_daily["selected_return"] - selected_daily["universe_return"]
    )
    return selected_daily


def selection_summary(frame: pd.DataFrame, selection: str) -> dict[str, float]:
    daily = daily_selection_outcomes(frame, selection)
    selected = selected_rows(frame, selection)
    return {
        "selection": selection,
        "selected_dates": len(daily),
        "selected_rows": len(selected),
        "mean_return": daily["selected_return"].mean(),
        "median_return": daily["selected_return"].median(),
        "positive_fraction": daily["selected_return"].gt(0).mean(),
        "mean_actual_target": daily["selected_target"].mean(),
        "mean_universe_return": daily["universe_return"].mean(),
        "mean_return_spread": daily["return_spread"].mean(),
        "median_return_spread": daily["return_spread"].median(),
        "mean_selected_count": daily["selected_count"].mean(),
    }

## 8. Fit the complete validation model matrix

In [ ]:
run_specs = list(product(TARGET_FAMILIES, HORIZONS, MODEL_NAMES))
metric_rows = []
prediction_frames = {}
fitted_models = {}

for family, horizon, model_name in tqdm(run_specs, desc="Short-horizon model matrix"):
    X_train, y_train, train_frame, X_validation, _, validation_frame = (
        make_experiment_frames(family, horizon)
    )

    started = time.perf_counter()
    try:
        model, predictions = fit_and_predict(
            model_name,
            X_train,
            y_train,
            train_frame,
            X_validation,
        )
    except Exception as error:
        raise RuntimeError(
            "Failed model run: "
            f"target_family={family}, horizon={horizon}, model={model_name}"
        ) from error

    elapsed_seconds = time.perf_counter() - started
    prediction_frame = add_predictions(validation_frame, predictions)
    key = (family, horizon, model_name)
    prediction_frames[key] = prediction_frame

    if STORE_FITTED_MODELS:
        fitted_models[key] = model

    metric_rows.append(
        {
            "target_family": family,
            "horizon": horizon,
            "model": model_name,
            "elapsed_seconds": elapsed_seconds,
            **aggregate_metrics(prediction_frame, model_name=model_name),
        }
    )

    if not STORE_FITTED_MODELS:
        del model
    gc.collect()

model_results = (
    pd.DataFrame(metric_rows)
    .set_index(["target_family", "horizon", "model"])
    .sort_index()
)
model_results

The locked test split has still not been requested. All model comparison and shortlisting below use
validation rows only.

## 9. Fixed top-k and complete top-score-bucket results

In [ ]:
selection_names = tuple(f"top_{k}" for k in TOP_K_VALUES) + ("top_score_bucket",)
selection_rows = []

for (family, horizon, model_name), frame in tqdm(
    prediction_frames.items(),
    desc="Tie-aware selection evaluation",
):
    for selection in selection_names:
        selection_rows.append(
            {
                "target_family": family,
                "horizon": horizon,
                "model": model_name,
                **selection_summary(frame, selection),
            }
        )

selection_results = (
    pd.DataFrame(selection_rows)
    .set_index(["target_family", "horizon", "model", "selection"])
    .sort_index()
)
selection_results

In [ ]:
top_1_comparison = (
    selection_results.xs("top_1", level="selection")
    .loc[:, [
        "selected_dates",
        "mean_return",
        "median_return",
        "positive_fraction",
        "mean_return_spread",
    ]]
    .sort_values("mean_return_spread", ascending=False)
)
top_1_comparison.head(15)

## 10. Validation subperiod stability

In [ ]:
subperiod_rows = []

for (family, horizon, model_name), frame in prediction_frames.items():
    for selection in selection_names:
        daily = daily_selection_outcomes(frame, selection)
        if daily.empty:
            continue
        dated = daily.copy()
        dated["year"] = pd.DatetimeIndex(dated.index).year
        for year, group in dated.groupby("year", sort=True):
            subperiod_rows.append(
                {
                    "target_family": family,
                    "horizon": horizon,
                    "model": model_name,
                    "selection": selection,
                    "year": int(year),
                    "dates": len(group),
                    "mean_return": group["selected_return"].mean(),
                    "median_return": group["selected_return"].median(),
                    "mean_return_spread": group["return_spread"].mean(),
                }
            )

subperiod_results = (
    pd.DataFrame(subperiod_rows)
    .set_index(["target_family", "horizon", "model", "selection", "year"])
    .sort_index()
)
subperiod_results

In [ ]:
subperiod_gate = (
    subperiod_results.reset_index()
    .groupby(["target_family", "horizon", "model", "selection"], sort=False)
    .agg(
        subperiod_count=("year", "nunique"),
        positive_spread_subperiods=(
            "mean_return_spread",
            lambda values: int(values.gt(0).sum()),
        ),
        minimum_subperiod_spread=("mean_return_spread", "min"),
        maximum_subperiod_spread=("mean_return_spread", "max"),
    )
)

candidate_table = selection_results.join(subperiod_gate)
candidate_table["all_subperiod_spreads_positive"] = (
    candidate_table["positive_spread_subperiods"]
    == candidate_table["subperiod_count"]
)

shortlist = candidate_table.loc[
    candidate_table["mean_return_spread"].gt(0)
    & candidate_table["median_return"].gt(0)
    & candidate_table["all_subperiod_spreads_positive"]
].sort_values(
    ["mean_return_spread", "positive_fraction"],
    ascending=[False, False],
).head(MAX_SHORTLIST_CONFIGURATIONS)

shortlist

### Decision gate

A configuration is provisionally shortlisted only when:

- its mean validation return spread is positive;

- its median selected return is positive; and

- its mean spread is positive in every validation year.

This gate is intentionally simple. Statistical significance is evaluated only for the highest-ranked
unique model/target/horizon configurations, and the locked test is still not opened.

## 11. Optional progress-visible bootstrap and permutation tests

In [ ]:
def bootstrap_selection_spreads(
    frame: pd.DataFrame,
    top_k_values=TOP_K_VALUES,
    *,
    iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = RANDOM_SEED,
    show_progress: bool = True,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []

    for k in tqdm(top_k_values, desc="Bootstrap top-k", disable=not show_progress):
        daily = daily_selection_outcomes(frame, f"top_{k}")
        spreads = daily["return_spread"].to_numpy(dtype="float64")
        estimates = np.empty(iterations, dtype="float64")
        for iteration in range(iterations):
            sampled = rng.choice(spreads, size=len(spreads), replace=True)
            estimates[iteration] = sampled.mean()

        rows.append(
            {
                "selection": f"top_{k}",
                "dates": len(spreads),
                "observed_mean_spread": spreads.mean(),
                "bootstrap_ci_low": np.quantile(estimates, 0.025),
                "bootstrap_ci_high": np.quantile(estimates, 0.975),
            }
        )
    return pd.DataFrame(rows).set_index("selection")


def permutation_groups(frame: pd.DataFrame):
    groups = []
    for _, group in group_by_date(frame, sort=True):
        eligible = eligible_daily_group(group)
        if eligible.empty or eligible["predicted_score"].nunique() == 1:
            continue
        groups.append(
            (
                eligible["predicted_score"].to_numpy(dtype="float64"),
                eligible["actual_return"].to_numpy(dtype="float64"),
                ticker_values(eligible),
            )
        )
    return groups


def within_date_permutation_tests(
    frame: pd.DataFrame,
    top_k_values=TOP_K_VALUES,
    *,
    iterations: int = PERMUTATION_ITERATIONS,
    seed: int = RANDOM_SEED,
    show_progress: bool = True,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    groups = permutation_groups(frame)
    if not groups:
        raise ValueError("No eligible non-flat dates remain for permutation testing.")

    observed = {
        k: daily_selection_outcomes(frame, f"top_{k}")["return_spread"].mean()
        for k in top_k_values
    }
    null_estimates = {
        k: np.empty(iterations, dtype="float64")
        for k in top_k_values
    }

    iterator = tqdm(
        range(iterations),
        desc="Within-date permutations",
        disable=not show_progress,
    )
    for iteration in iterator:
        daily_spreads = {k: [] for k in top_k_values}
        for scores, actual_returns, tickers in groups:
            permuted_scores = rng.permutation(scores)
            order = np.lexsort((tickers, -permuted_scores))
            universe_return = actual_returns.mean()
            for k in top_k_values:
                selected_return = actual_returns[order[: min(k, len(order))]].mean()
                daily_spreads[k].append(selected_return - universe_return)

        for k in top_k_values:
            null_estimates[k][iteration] = np.mean(daily_spreads[k])

    rows = []
    for k in top_k_values:
        null = null_estimates[k]
        observed_spread = observed[k]
        rows.append(
            {
                "selection": f"top_{k}",
                "observed_mean_spread": observed_spread,
                "null_mean_spread": null.mean(),
                "null_p95": np.quantile(null, 0.95),
                "one_sided_p_value": (
                    1 + np.sum(null >= observed_spread)
                ) / (iterations + 1),
            }
        )
    return pd.DataFrame(rows).set_index("selection")

In [ ]:
if RUN_RESAMPLING:
    unique_configurations = (
        shortlist.reset_index()
        .sort_values("mean_return_spread", ascending=False)
        .drop_duplicates(["target_family", "horizon", "model"])
        .head(RESAMPLING_CONFIGURATION_LIMIT)
    )

    if unique_configurations.empty:
        print("No configurations passed the validation shortlist gate; resampling was skipped.")
    else:
        resampling_rows = []
        for row in unique_configurations.itertuples(index=False):
            key = (row.target_family, row.horizon, row.model)
            frame = prediction_frames[key]
            bootstrap = bootstrap_selection_spreads(frame)
            permutation = within_date_permutation_tests(frame)
            combined = bootstrap.join(permutation, rsuffix="_permutation").reset_index()
            combined.insert(0, "model", row.model)
            combined.insert(0, "horizon", row.horizon)
            combined.insert(0, "target_family", row.target_family)
            resampling_rows.append(combined)

        resampling_results = pd.concat(resampling_rows, ignore_index=True)
        display(resampling_results)
else:
    print(
        "Resampling skipped. Review the shortlist first, then set RUN_RESAMPLING = True "
        "to test the leading unique configurations with visible progress."
    )

## 12. Compact comparison views

In [ ]:
plot_data = (
    selection_results.xs("top_1", level="selection")["mean_return_spread"]
    .rename("mean_return_spread")
    .reset_index()
)

for family in TARGET_FAMILIES:
    family_data = plot_data.loc[plot_data["target_family"].eq(family)]
    pivot = family_data.pivot(index="horizon", columns="model", values="mean_return_spread")
    ax = pivot.plot(marker="o", figsize=(10, 5))
    ax.axhline(0, linewidth=1)
    ax.set_title(f"Top-1 mean return spread by horizon — {family}")
    ax.set_xlabel("Horizon (sessions)")
    ax.set_ylabel("Selected return minus same-date universe return")
    plt.show()

In [ ]:
shortlist_columns = [
    "selected_dates",
    "mean_return",
    "median_return",
    "positive_fraction",
    "mean_return_spread",
    "minimum_subperiod_spread",
    "median_top_bucket_size",
    "flat_score_date_fraction",
    "mean_daily_spearman",
    "mean_daily_ndcg_at_5",
]

model_metric_columns = [
    "median_top_bucket_size",
    "flat_score_date_fraction",
    "mean_daily_spearman",
    "mean_daily_ndcg_at_5",
]
shortlist_with_model_metrics = (
    shortlist.reset_index()
    .merge(
        model_results.loc[:, model_metric_columns].reset_index(),
        on=["target_family", "horizon", "model"],
        how="left",
        validate="many_to_one",
    )
    .set_index(["target_family", "horizon", "model", "selection"])
)
shortlist_with_model_metrics.loc[:, shortlist_columns]

## 13. Interpretation and handoff

Use the completed outputs to make one bounded decision:

- Advance at most two configurations if their top-k or complete top-score-bucket spread is
positive, stable in both validation years, and survives the matched permutation test.

- Retain Ridge only as a control , not as a candidate, unless it unexpectedly demonstrates
stable ranking value.

- Do not select a model from aggregate RMSE or (R^2) alone. The production decision is
cross-sectional selection, so daily ranking and selected-return spread are primary.

- Treat flat-score dates as no-trade dates. Never force a recommendation where the model has
no cross-sectional distinction.

- Treat a tied maximum as a candidate bucket. Evaluate the complete bucket rather than
pretending ticker-order tie-breaking is model information.

- Do not open the locked test during this notebook.

After this matrix is complete, the next notebook should be 04_excursion_and_barrier_analysis.ipynb . It should take no more than the two leading
configurations and test whether their predicted opportunity can be converted into attractive
MFE/MAE and stop-before-target behavior. Universe expansion and neural sequence models remain
separate later experiments.

## 14. Prediction-shape diagnostics

In [ ]:
ncols = 3
nsubplots = len(prediction_frames)
nrows = int(np.ceil(nsubplots / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(13, 2.6 * nrows),
    layout="constrained",
)
axes = np.asarray(axes).reshape(-1)

for ax, (key, frame) in zip(axes, prediction_frames.items()):
    ax.hexbin(
        x=frame["predicted_score"],
        y=frame["actual_return"],
        mincnt=1,
        bins="log",
        gridsize=200,
        cmap="terrain",
    )
    ax.grid()
    ax.set_title(" | ".join(str(part) for part in key), fontsize=11)
    ax.set_xlabel("Predicted score")
    ax.set_ylabel("Realized raw return")

for ax in axes[len(prediction_frames):]:
    ax.set_visible(False)

plt.show()

In [ ]:
prediction_shape_summary = (
    model_results.loc[
        :,
        [
            "prediction_mean",
            "prediction_std",
            "global_unique_predictions",
            "median_daily_unique_predictions",
            "flat_score_date_fraction",
            "median_top_bucket_size",
        ],
    ]
    .sort_index()
)
prediction_shape_summary

In [ ]:

actual_col = "actual_return"
values = prediction_frames[("raw_return", 4, "xgboost_regressor")].query("predicted_score > 0.05")[actual_col]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    values,
    bins=1000,
    histtype="step",
    linewidth=2,
    cumulative=True,
    density=True,
    log=False,
    zorder=3,
)
ax.grid(zorder=3, which="both")
ax.axvline(0, color="k", linewidth=1, zorder=2)
ax.set_xticks(np.arange(-0.4, 0.9, 0.1))
ax.set_yticks(np.arange(0, 1.01, 0.1))
ax.set_xlabel(actual_col, fontweight="bold")
ax.set_ylabel("Cumulative density", fontweight="bold")
plt.show()

In [ ]:
(
    prediction_frames[("raw_return", 5, "xgboost_ranker")][["actual_return", "predicted_score"]]
    .sort_values(["predicted_score"], ascending=False)
    .head(20)
)

In [ ]:
prediction_frames[("atr_units", 2, "xgboost_ranker")].query("predicted_score > 1")[["actual_return", "predicted_score"]]

In [ ]:
temp = pd.concat(
    [
        prediction_frames[("raw_return", 2, "xgboost_regressor")]["actual_return"],
        prediction_frames[("raw_return", 2, "xgboost_regressor")]["predicted_score"].rename("reg_pred"),
        prediction_frames[("raw_return", 2, "xgboost_ranker")]["predicted_score"].rename("rank_pred"),
    ],
    axis=1,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    x=temp["reg_pred"],
    y=temp["rank_pred"],
    c=temp["actual_return"],
    s=10,
    alpha=0.5,
    vmin=-0.1, vmax=0.1,
    cmap="seismic",
    zorder=3,
)
ax.grid(zorder=3)
plt.show()

In [ ]:
temp = pd.concat(
    [
        prediction_frames[("raw_return", 2, "xgboost_regressor")]["actual_return"],
        prediction_frames[("raw_return", 2, "xgboost_regressor")]["predicted_score"].rename("reg_pred"),
        prediction_frames[("raw_return", 2, "xgboost_ranker")]["predicted_score"].rename("rank_pred"),
    ],
    axis=1,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hexbin(
    x=temp["reg_pred"],
    y=temp["rank_pred"],
    mincnt=1,
    bins="log",
    gridsize=200,
    cmap="terrain",
    zorder=3,
)
ax.grid(zorder=3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    temp.query("reg_pred > 0.04 & rank_pred > 0.5")["actual_return"],
    bins=1000,
    histtype="step",
    linewidth=1.5,
    cumulative=True,
    density=True,
    log=False,
    zorder=3,
)
ax.grid(zorder=1, which="both")
ax.axhline(0.5, color="k", linewidth=1, zorder=2)
ax.axvline(0, color="k", linewidth=1, zorder=2)
ax.set_yticks(np.arange(0, 1.01, 0.1))
ax.set_xticks(np.arange(-0.4, 1, 0.1))
plt.show()

# Run backtest

In [ ]:
from swingtrader.modeling.backtest import run_backtest
from swingtrader.indicators import atr

In [ ]:
INITIAL_CASH = 50000
RISK_FRACTION = 0.005
MAX_POSITIONS = 10
MAX_HOLDING_SESSIONS = 5
MIN_SCORE = 0.01
STOP_ATR_MULTIPLE = 1.0
REWARD_RISK_RATIO = 2.0
COMMISSION_RATE = 0.0025

In [ ]:
all_backtest_results = dict()

for k, v in prediction_frames.items():
    label, horizon, mdl = k
    selected_score = v["predicted_score"].rename("score")
    backtest_prices = prices.loc[selected_score.index, ["open", "high", "low", "close"]]
    signals = pd.concat([selected_score, atr(backtest_prices)], axis=1)

    first_signal_date = signals.index.get_level_values("trading_date").min()
    minimum_score = np.quantile(selected_score, 0.999)

    backtest_result = run_backtest(
        backtest_prices,
        signals,
        initial_cash=INITIAL_CASH,
        risk_fraction=RISK_FRACTION,
        max_positions=MAX_POSITIONS,
        max_holding_sessions=MAX_HOLDING_SESSIONS,
        minimum_score=minimum_score,
        stop_atr_multiple=STOP_ATR_MULTIPLE,
        reward_risk_ratio=REWARD_RISK_RATIO,
        commission_rate=COMMISSION_RATE,
    )
    all_backtest_results[k] = backtest_result

    trades = backtest_result["trades"]
    equity = backtest_result["equity"]
    summary = backtest_result["summary"]

    print(f"\n\n{k}")
    print(f"minimum_score = {minimum_score}")
    print(f"Completed trades: {len(trades):,}")
    print(summary)

In [ ]:
selected_score = prediction_frames[("raw_return", 5, "ridge")]["predicted_score"].rename("score")
backtest_prices = prices.loc[selected_score.index, ["open", "high", "low", "close"]]
signals = pd.concat([selected_score, atr(backtest_prices)], axis=1)

first_signal_date = signals.index.get_level_values("trading_date").min()

backtest_result = run_backtest(
    backtest_prices,
    signals,
    initial_cash=INITIAL_CASH,
    risk_fraction=RISK_FRACTION,
    max_positions=MAX_POSITIONS,
    max_holding_sessions=MAX_HOLDING_SESSIONS,
    minimum_score=0.01,
    stop_atr_multiple=STOP_ATR_MULTIPLE,
    reward_risk_ratio=REWARD_RISK_RATIO,
    commission_rate=COMMISSION_RATE,
)

trades = backtest_result["trades"]
equity = backtest_result["equity"]
summary = backtest_result["summary"]

print(f"Completed trades: {len(trades):,}")
display(summary.to_frame("value"))

In [ ]:
equity = all_backtest_results[('raw_return', 3, 'ridge')]["equity"]

ax = equity["equity"].plot(
    figsize=(12, 5),
    title="Backtest portfolio equity",
    ylabel="Equity (SEK)",
)
ax.set_xlabel("Trading date")
ax.grid(True)
plt.show()

display(equity.tail(10))

In [ ]:
equity.reset_index().describe()